In [21]:
#Bu notebookda eğitilen 2.model ile BoT-SORT Re-ID yöntemi denenecektir.
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') or f.endswith('.mp4'):
            print(os.path.join(root, f))

/kaggle/input/datasets/zlemdemir/atolye-video/fabrika_video.mp4
/kaggle/input/datasets/zlemdemir/2-model/2_operator_shadowing_model.pt


In [22]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.1 MB/s eta 0:00:00


In [23]:
import shutil
import os

#varsayılan olarak Re-ID özelliği kapalı bunu açmak için yolu kontrol ediyoruz : with_reid: True yapacağız.
# Ultralytics'in varsayılan botsort.yaml dosyasını bul
import ultralytics
ultra_path = os.path.dirname(ultralytics.__file__)
default_yaml = os.path.join(ultra_path, 'cfg', 'trackers', 'botsort.yaml')

print("Varsayılan yaml yolu:", default_yaml)

# İçeriğini görelim
with open(default_yaml, 'r') as f:
    content = f.read()
print("\nVarsayılan içerik:")
print(content)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Varsayılan yaml yolu: /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/trackers/botsort.yaml

Varsayılan içerik:
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# BoT-SORT tracker defaults for mode="track"
# Docs: https://docs.ultralytics.com/modes/track

tracker_type: botsort # (str) Tracker backend: botsort
track_high_thresh: 0.25 # (float) First-stage match threshold; raise for cleaner tracks, lower to keep more
track_low_thresh: 0.1 # (float) Second-stage threshold for low-score matches; balances recovery vs drift
new_track_thresh: 0.25 # (float) Start a new track if no match ≥ this; higher reduces false tracks
track_buffer: 30 # (int) Frames to keep

In [27]:
# Yeni yaml - gerçek Re-ID modeli ile
reid_yaml_v2 = """tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: 90
match_thresh: 0.8
fuse_score: True

gmc_method: sparseOptFlow

proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: True
model: yolo11n-cls.pt
"""

with open('/kaggle/working/botsort_reid_v2.yaml', 'w') as f:
    f.write(reid_yaml_v2)

print("botsort_reid_v2.yaml oluşturuldu - gerçek Re-ID modeli ile")

botsort_reid_v2.yaml oluşturuldu - gerçek Re-ID modeli ile


In [30]:
from ultralytics import YOLO
import pandas as pd

model_path = '/kaggle/input/datasets/zlemdemir/2-model/2_operator_shadowing_model.pt'
video_path = '/kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4'

# --- 1. Varsayılan BoT-SORT (Re-ID kapalı) ---
print("=" * 50)
print("1. Varsayılan BoT-SORT (Re-ID KAPALI)")
print("=" * 50)

model = YOLO(model_path)
results_default = model.track(
    source=video_path,
    tracker='botsort.yaml',
    save=True,
    conf=0.5,
    project='/kaggle/working',
    name='botsort_default'
)

# ID istatistikleri
ids_default = set()
for result in results_default:
    if result.boxes.id is not None:
        ids_default.update(result.boxes.id.cpu().numpy().astype(int).tolist())

print(f"\nToplam benzersiz ID sayısı (Re-ID kapalı): {len(ids_default)}")
print(f"ID'ler: {sorted(ids_default)}")

1. Varsayılan BoT-SORT (Re-ID KAPALI)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 927.6ms
video 1/1 (frame 2/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 927.6ms
video 1/1 (frame 3/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 931.4ms
video 1/1 (frame 4

In [31]:
print("=" * 50)
print("2. BoT-SORT + Re-ID AÇIK")
print("=" * 50)

model = YOLO(model_path)
results_reid = model.track(
    source=video_path,
    tracker='/kaggle/working/botsort_reid.yaml',
    save=True,
    conf=0.5,
    project='/kaggle/working',
    name='botsort_reid'
)

# ID istatistikleri
ids_reid = set()
for result in results_reid:
    if result.boxes.id is not None:
        ids_reid.update(result.boxes.id.cpu().numpy().astype(int).tolist())

print(f"\nToplam benzersiz ID sayısı (Re-ID açık): {len(ids_reid)}")
print(f"ID'ler: {sorted(ids_reid)}")

print("\n" + "=" * 50)
print("KARŞILAŞTIRMA")
print("=" * 50)
print(f"Re-ID kapalı: 5 ID (1, 2, 5, 6, 7)")
print(f"Re-ID açık: {len(ids_reid)} ID ({sorted(ids_reid)})")

2. BoT-SORT + Re-ID AÇIK

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 888.0ms
video 1/1 (frame 2/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 915.7ms
video 1/1 (frame 3/331) /kaggle/input/datasets/zlemdemir/video-for-tracking-3-1/video_for_tracking_3.mp4: 736x1280 5 persons, 873.4ms
video 1/1 (frame 4/331) /kaggle